# FastAPI: Modern Web Framework

**Purpose:** Build fast, modern REST APIs with automatic documentation.

← [12. Flask](./12-flask.ipynb) | [Modules](./README.md) | **13. FastAPI** | [14. Django →](./14-django.ipynb)

## Simple: Basic API

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class User(BaseModel):
    id: int
    name: str
    email: str

@app.get("/")
def read_root():
    return {"message": "Hello, FastAPI!"}

@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"id": user_id, "name": f"User {user_id}"}

@app.post("/users")
def create_user(user: User):
    return {"status": "created", "user": user}

print("FastAPI app defined")

## Medium: Query Parameters & Validation

In [ ]:
from fastapi import FastAPI, Query
from pydantic import BaseModel, Field

app = FastAPI()

class Product(BaseModel):
    name: str = Field(..., min_length=1)
    price: float = Field(..., gt=0)
    quantity: int = Field(default=1, ge=1)

@app.get("/search")
def search(q: str = Query(..., min_length=1), skip: int = 0, limit: int = 10):
    # GET /search?q=widget&skip=0&limit=10
    return {"query": q, "skip": skip, "limit": limit}

@app.post("/products")
def create_product(product: Product):
    # Validates: name (required, string), price (>0), quantity (>=1)
    return {"status": "created", "product": product}

print("FastAPI with validation defined")

## Complex: Full CRUD with Models

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    description: str = None
    price: float

class ItemInDB(Item):
    id: int

db = {1: {"id": 1, "name": "Widget", "description": "A widget", "price": 9.99}}

@app.get("/items/{item_id}", response_model=ItemInDB)
async def get_item(item_id: int):
    if item_id not in db:
        raise HTTPException(status_code=404, detail="Item not found")
    return db[item_id]

@app.post("/items", response_model=ItemInDB, status_code=201)
async def create_item(item: Item):
    item_id = max(db.keys()) + 1 if db else 1
    db[item_id] = {"id": item_id, **item.model_dump()}
    return db[item_id]

@app.put("/items/{item_id}", response_model=ItemInDB)
async def update_item(item_id: int, item: Item):
    if item_id not in db:
        raise HTTPException(status_code=404, detail="Item not found")
    db[item_id].update(item.model_dump())
    return db[item_id]

@app.delete("/items/{item_id}")
async def delete_item(item_id: int):
    if item_id not in db:
        raise HTTPException(status_code=404, detail="Item not found")
    del db[item_id]
    return {"message": "Item deleted"}

print("FastAPI CRUD app defined")

## Testing with TestClient (No Server Needed)

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float

items = {1: {"name": "Widget", "price": 9.99}}

@app.get("/items/{item_id}")
async def get_item(item_id: int):
    if item_id not in items:
        return {"error": "Not found"}
    return items[item_id]

@app.post("/items")
async def create_item(item: Item):
    item_id = max(items.keys()) + 1
    items[item_id] = item.model_dump()
    return {"id": item_id, **item.model_dump()}

# Test using TestClient (no server needed)
client = TestClient(app)

response = client.get("/items/1")
print("GET /items/1:", response.json())

new_item = {"name": "Gadget", "price": 19.99}
response = client.post("/items", json=new_item)
print("POST /items:", response.json())

---

## What is UV and Uvicorn?
**UV:** Modern Python package manager (faster than pip, written in Rust)
- Install packages: `uv pip install fastapi uvicorn`
- Run scripts directly: `uv run script.py` (no venv activation)
- Combined: `uv run --with fastapi --with uvicorn uvicorn app:app`

**Uvicorn:** ASGI web server for running FastAPI (and other async Python web frameworks)
- FastAPI requires Uvicorn to run (or similar ASGI server)
- Replaces Flask's built-in server (which is synchronous)
- Supports async/await: `async def endpoint()`
- Command: `uvicorn app:app --reload --port 8000`

Flask uses its own server (`app.run()`), but FastAPI uses Uvicorn (`uvicorn app:app`).

---

## How to Run & Call FastAPI

### Setup: Using Python or UV
```bash
# With traditional Python + pip:
python -m venv venv
source venv/bin/activate  # On Windows: venv\Scripts\activate
pip install fastapi uvicorn

# With UV (faster, simpler):
uv venv
source .venv/bin/activate  # On Windows: .venv\Scripts\activate
uv pip install fastapi uvicorn
# OR in one command: uv run --with fastapi --with uvicorn uvicorn app:app --reload
```

### Option 1: Run FastAPI Server (Custom Ports)
```bash
# Default port (8000) - save app to app.py:

# With Python (default 8000):
uvicorn app:app --reload

# With Python (custom port 3000):
uvicorn app:app --reload --port 3000 --host 0.0.0.0

# With Python (custom port 9000):
uvicorn app:app --reload --port 9000

# With UV (default 8000):
uv run --with fastapi --with uvicorn uvicorn app:app --reload

# With UV (custom port 3000):
uv run --with fastapi --with uvicorn uvicorn app:app --reload --port 3000 --host 0.0.0.0

# With UV (custom port 9000):
uv run --with fastapi --with uvicorn uvicorn app:app --reload --port 9000

# Visit: http://localhost:8000/docs (default) or http://localhost:3000/docs (custom)
# Auto-generated Swagger UI available at /docs, ReDoc at /redoc
```

### Custom Port in Python Code
```python
from fastapi import FastAPI
import uvicorn

app = FastAPI()

@app.get('/')
def read_root():
    return {'message': 'Hello'}

if __name__ == '__main__':
    # Default: port 8000
    uvicorn.run(app, host='0.0.0.0', port=8000)
    
    # Custom port: 3000
    uvicorn.run(app, host='0.0.0.0', port=3000, reload=True)
    
    # Custom port from environment variable:
    import os
    port = int(os.getenv('FASTAPI_PORT', 8000))
    uvicorn.run(app, host='0.0.0.0', port=port, reload=True)
```

### Run on Custom Port
```bash
# Using environment variable:
# With Python:
export FASTAPI_PORT=3000  # On Windows: set FASTAPI_PORT=3000
uvicorn app:app --reload
# Note: Command-line --port overrides env var

# With UV:
FASTAPI_PORT=3000 uv run --with fastapi --with uvicorn uvicorn app:app --reload
# On Windows: set FASTAPI_PORT=3000 && uv run --with fastapi --with uvicorn uvicorn app:app --reload
```

### Option 2: Call FastAPI from Python (Sync - Requests)
```python
import requests

BASE_URL = "http://localhost:8000"

# GET request
response = requests.get(f"{BASE_URL}/users/1")
print(response.json())  # {"id": 1, "name": "User 1"}

# GET with query parameters
response = requests.get(f"{BASE_URL}/search", params={"q": "widget", "skip": 0})
print(response.json())

# POST request (create)
payload = {"name": "Bob", "email": "bob@example.com"}
response = requests.post(f"{BASE_URL}/users", json=payload)
print(response.json(), response.status_code)  # 201 Created

# PUT request (update)
updates = {"name": "Bob Updated", "email": "bob.new@example.com"}
response = requests.put(f"{BASE_URL}/users/1", json=updates)
print(response.json())

# DELETE request
response = requests.delete(f"{BASE_URL}/users/1")
print(response.status_code)  # 204 No Content
```

### Option 3: Call FastAPI from Python (Async - AIOHTTP)
```python
import asyncio
from aiohttp import ClientSession

async def call_api():
    async with ClientSession() as session:
        # GET
        async with session.get('http://localhost:8000/users/1') as resp:
            data = await resp.json()
            print(data)
        
        # POST
        payload = {"name": "Alice", "email": "alice@example.com"}
        async with session.post('http://localhost:8000/users', json=payload) as resp:
            data = await resp.json()
            print(data)

# In Jupyter: await call_api()
# In script: asyncio.run(call_api())
```

### Option 4: Test in Jupyter (No Server Needed)
```python
from fastapi.testclient import TestClient

client = TestClient(app)  # Test client (no server)

# Test GET
response = client.get("/users/1")
print(response.json())

# Test POST
payload = {"name": "Charlie", "email": "charlie@example.com"}
response = client.post("/users", json=payload)
print(response.json())

# Test PUT
response = client.put("/users/1", json={"name": "Charlie Updated"})
print(response.status_code)

# Test DELETE
response = client.delete("/users/1")
print(response.status_code)
```

### Option 5: Access Auto-Generated Documentation
- **Swagger UI:** http://localhost:8000/docs
- **ReDoc:** http://localhost:8000/redoc
- **OpenAPI JSON:** http://localhost:8000/openapi.json

### Status Codes
- `200` OK - Success
- `201` Created - POST success
- `204` No Content - DELETE success
- `400` Bad Request - Validation error
- `404` Not Found - Resource missing
- `422` Unprocessable Entity - Invalid data
- `500` Internal Server Error

## Resources

- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [FastAPI Tutorial](https://fastapi.tiangolo.com/tutorial/)
